# Cyclical EMA Strategy Research

Goal: 在周期性波动美股标的上，验证 EMA 趋势跟踪 + LightGBM gating 是否产生 Sharpe ≥ 1.0 且优于纯 EMA baseline ≥ +0.3 的策略。

Reference:
- Design: `docs/plans/2026-05-07-cyclical-ema-research-design.md`
- Impl plan: `docs/plans/2026-05-08-cyclical-ema-research-impl.md`

In [1]:
# === Cell 0: Imports + Constants ===
from __future__ import annotations
import os, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import optuna
import shap

from oxq.indicators.builtin import EMA, ATR, MFI, OBV
from oxq.indicators.hurst_exponent import HurstExponent
from oxq.indicators.annualized_volatility import AnnualizedVolatility
from oxq.indicators.rolling_volatility import RollingVolatility
from oxq.indicators.garch_volatility import GarchVolatility
from oxq.indicators.rolling_mdd import RollingMDD
from oxq.indicators.nday_return import NdayReturn

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

# --- Constants ---
def _find_repo_root() -> Path:
    """Walk up from cwd until we find pyproject.toml (the open-xquant repo marker)."""
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(
        "Cannot find repo root (no pyproject.toml in cwd or any parent). "
        "Run the notebook from somewhere inside the open-xquant repo tree."
    )

REPO_ROOT    = _find_repo_root()
PROJECT_ROOT = REPO_ROOT / "examples" / "research" / "cyclical_ema"
CACHE_DIR    = PROJECT_ROOT / "cache"
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# 数据刷新 flag —— 默认 False，迭代时使用缓存；改 True 强制重拉
REFRESH_DATA = False

# 时间锚点（来自 design doc 第 2.3 节）
ANCHOR_DATE   = pd.Timestamp("2026-05-07")            # universe filter as-of
TRAIN_CUTOFF  = pd.Timestamp("2021-05-07")            # 训练/测试切分
HISTORY_START = pd.Timestamp("2010-01-01")            # 数据起点（覆盖 ≥10y 的 universe）

# Universe 过滤阈值
MIN_LISTING_YEARS_STRICT = 10                          # A 组
MIN_LISTING_YEARS_RELAX  = 7                           # B 组
MIN_DOLLAR_VOL_STRICT    = 5_000_000                   # A 组
MIN_DOLLAR_VOL_RELAX     = 2_000_000                   # B 组
HURST_WINDOW             = 100
HURST_THRESHOLD_STRICT   = 0.40
HURST_THRESHOLD_RELAX    = 0.50
ANN_VOL_WINDOW           = 60
ANN_VOL_THRESHOLD_STRICT = 0.25
ANN_VOL_THRESHOLD_RELAX  = 0.20

# Sample 标签
LABEL_RETURN_THRESHOLD   = 0.05                        # gross_return ≥ 5% → 1
MIN_HISTORY_BARS         = 252                         # t_buy 之前最少交易日数

# 模型 + 回测
RANDOM_STATE             = 42
N_OPTUNA_TRIALS          = 50
N_CV_FOLDS               = 5
SCORE_THRESHOLD_X        = 0.55
TOP_N                    = 10
ROUND_TRIP_BPS           = 20                          # 0.20%

# 行业映射
SECTOR_TO_SPDR = {
    "Technology":              "XLK",
    "Financial Services":      "XLF",
    "Energy":                  "XLE",
    "Healthcare":              "XLV",
    "Industrials":             "XLI",
    "Consumer Defensive":      "XLP",
    "Consumer Cyclical":       "XLY",
    "Utilities":               "XLU",
    "Basic Materials":         "XLB",
    "Real Estate":             "XLRE",
    "Communication Services":  "XLC",
}
MARKET_TICKERS = ["^GSPC", "^VIX"] + list(set(SECTOR_TO_SPDR.values()))

print(f"Project root: {PROJECT_ROOT}")
print(f"REFRESH_DATA: {REFRESH_DATA}")
print(f"Anchor date: {ANCHOR_DATE.date()}, Train cutoff: {TRAIN_CUTOFF.date()}")


Project root: /Users/daodao/Documents/2-coding-space/git/github.com/open-xquant/examples/research/cyclical_ema
REFRESH_DATA: False
Anchor date: 2026-05-07, Train cutoff: 2021-05-07


In [2]:
# === Cell 1: Universe ticker list (Nasdaq Trader) ===
TICKER_CACHE = CACHE_DIR / "nasdaq_tickers.parquet"

def fetch_nasdaq_tickers() -> pd.DataFrame:
    """Fetch nasdaqlisted.txt + otherlisted.txt; return cleaned common-stock tickers."""
    base = "https://www.nasdaqtrader.com/dynamic/symdir"
    nq = pd.read_csv(f"{base}/nasdaqlisted.txt", sep="|")
    nq = nq[nq["Symbol"].notna()].copy()                       # was: notna() & != "File Creation Time"
    nq["Exchange"] = "NASDAQ"
    nq = nq[(nq["Test Issue"] == "N") & (nq["ETF"] == "N")]

    ot = pd.read_csv(f"{base}/otherlisted.txt", sep="|")
    ot = ot[ot["ACT Symbol"].notna()].copy()
    ot = ot.rename(columns={"ACT Symbol": "Symbol"})
    ot = ot[(ot["Test Issue"] == "N") & (ot["ETF"] == "N")]

    df = pd.concat(
        [nq[["Symbol", "Security Name", "Exchange"]],
         ot[["Symbol", "Security Name", "Exchange"]]],
        ignore_index=True,
    )

    # 排除 ADR / 衍生品（基于 Symbol 字符 + Security Name 关键词）
    # 故意 NOT 用 "Symbol len > 4" —— 会误杀 GOOGL / CMCSA / LBRDK 等大盘股 class share
    bad_symbol = df["Symbol"].str.contains(r"[\.\$=\^]", regex=True, na=False)
    derivative_name = df["Security Name"].str.contains(
        r"Warrant|Preferred|Units?\b|Rights?\b|Note|Convertible|Bond|Depositary|ADR|ADS",
        case=False, regex=True, na=False)
    # 显式去掉 "File Creation Time: ..." 页脚行（startswith，带时间戳）
    footer = df["Symbol"].str.startswith("File Creation Time", na=False)

    df = df[~bad_symbol & ~derivative_name & ~footer].reset_index(drop=True)
    return df

if REFRESH_DATA or not TICKER_CACHE.exists():
    tickers_df = fetch_nasdaq_tickers()
    tickers_df.to_parquet(TICKER_CACHE)
else:
    tickers_df = pd.read_parquet(TICKER_CACHE)

print(f"Total tickers after type filter: {len(tickers_df)}")
print(tickers_df.head(10))

assert 4000 < len(tickers_df) < 7000, f"Expected 4000-7000 tickers, got {len(tickers_df)}"
assert tickers_df["Symbol"].is_unique, "Symbols should be unique"


Total tickers after type filter: 5510
  Symbol                                      Security Name Exchange
0   AACB  Artius II Acquisition Inc. - Class A Ordinary ...   NASDAQ
1   AACI  Armada Acquisition Corp. III - Class A Ordinar...   NASDAQ
2   AACO  Abony Acquisition Corp. I - Class A Ordinary S...   NASDAQ
3    AAL       American Airlines Group, Inc. - Common Stock   NASDAQ
4   AAME       Atlantic American Corporation - Common Stock   NASDAQ
5   AAOI       Applied Optoelectronics, Inc. - Common Stock   NASDAQ
6   AAON                          AAON, Inc. - Common Stock   NASDAQ
7   AAPG  Ascentage Pharma Group International - America...   NASDAQ
8   AAPL                          Apple Inc. - Common Stock   NASDAQ
9   AARD         Aardvark Therapeutics, Inc. - Common Stock   NASDAQ


In [3]:
# === Cell 2: Wikipedia 历史指数变更（C 组退市票来源）===
DELISTED_CACHE = CACHE_DIR / "delisted_tickers.parquet"

def scrape_sp500_changes() -> pd.DataFrame:
    """Wikipedia 'List of S&P 500 companies' has a 'Selected changes' table
    listing index removals (date, ticker, security)."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0 (research/educational)"}
    html = requests.get(url, headers=headers, timeout=30).text
    tables = pd.read_html(html)
    # 第二张表通常是 "Selected changes to the list of S&P 500 components"
    changes = tables[1]
    # 列结构兼容 multi-index header
    changes.columns = [' '.join(c).strip() if isinstance(c, tuple) else c for c in changes.columns]
    rem_col = [c for c in changes.columns if "Removed" in c and ("Ticker" in c or "Symbol" in c)]
    if not rem_col:
        return pd.DataFrame(columns=["ticker", "removed_date"])
    df = changes[[changes.columns[0], rem_col[0]]].copy()
    df.columns = ["removed_date", "ticker"]
    df = df.dropna(subset=["ticker"])
    df["removed_date"] = pd.to_datetime(df["removed_date"], errors="coerce")
    df = df.dropna(subset=["removed_date"])
    df["ticker"] = df["ticker"].astype(str).str.strip()
    df = df[df["ticker"].str.match(r"^[A-Z]{1,4}$", na=False)]
    return df

if REFRESH_DATA or not DELISTED_CACHE.exists():
    delisted_df = scrape_sp500_changes()
    # 仅保留在 HISTORY_START 之后被剔出的（之前的太老，yfinance 多半拉不到数据）
    delisted_df = delisted_df[delisted_df["removed_date"] >= HISTORY_START].reset_index(drop=True)
    # 去重（同一票可能被剔出多次）
    delisted_df = delisted_df.sort_values("removed_date").drop_duplicates("ticker", keep="first")
    delisted_df.to_parquet(DELISTED_CACHE)
else:
    delisted_df = pd.read_parquet(DELISTED_CACHE)

# 排除当前还在 universe 里的票（这些不算"退市"）
current_set = set(tickers_df["Symbol"].tolist())
delisted_df = delisted_df[~delisted_df["ticker"].isin(current_set)].reset_index(drop=True)

print(f"Historical removals (post-{HISTORY_START.date()}, not in current universe): {len(delisted_df)}")
print(delisted_df.head(10))

assert 100 < len(delisted_df) < 2000, f"Expected 100-2000 historical removals, got {len(delisted_df)}"


Historical removals (post-2010-01-01, not in current universe): 189
  removed_date ticker
0   2010-02-26     RX
1   2010-04-29    BJS
2   2010-06-28    XTO
3   2010-06-30    STR
4   2010-07-14    MIL
5   2010-11-17    PTV
6   2010-12-17    ODP
7   2010-12-17     EK
8   2011-01-03    MDP
9   2011-02-25    AYE


In [4]:
# === Cell 3: 价格数据下载（yfinance）===
PRICE_CACHE = CACHE_DIR / "prices.parquet"
ERROR_LOG   = CACHE_DIR / "data_errors.log"
BATCH_SIZE  = 200  # yfinance batch 上限观测值

all_tickers = list(set(tickers_df["Symbol"].tolist() + delisted_df["ticker"].tolist()))
print(f"Total tickers to download: {len(all_tickers)}")

def download_prices(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    """Batch download with error logging. Returns long-format DataFrame."""
    all_data = []
    errors = []
    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i+BATCH_SIZE]
        try:
            data = yf.download(
                tickers=" ".join(batch),
                start=start, end=end,
                group_by="ticker",
                auto_adjust=False,  # 我们手动用 Adj Close
                progress=False, threads=True,
            )
        except Exception as e:
            errors.append((batch, str(e)))
            continue
        for t in batch:
            try:
                if t in data.columns.get_level_values(0):
                    df = data[t].dropna(how="all").copy()
                    if df.empty:
                        errors.append((t, "empty"))
                        continue
                    df["ticker"] = t
                    df = df.reset_index().rename(columns={"Date": "date"})
                    all_data.append(df)
                else:
                    errors.append((t, "missing"))
            except Exception as e:
                errors.append((t, str(e)))
        print(f"  batch {i//BATCH_SIZE + 1}/{(len(tickers)+BATCH_SIZE-1)//BATCH_SIZE} done")

    with open(ERROR_LOG, "a") as f:
        f.write(f"\n--- Run at {datetime.now()} ---\n")
        for t, err in errors:
            f.write(f"{t}: {err}\n")

    if not all_data:
        return pd.DataFrame()
    return pd.concat(all_data, ignore_index=True)

if REFRESH_DATA or not PRICE_CACHE.exists():
    prices = download_prices(
        all_tickers,
        start=HISTORY_START.strftime("%Y-%m-%d"),
        end=(ANCHOR_DATE + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    )
    prices = prices.set_index(["ticker", "date"]).sort_index()
    prices.to_parquet(PRICE_CACHE)
else:
    prices = pd.read_parquet(PRICE_CACHE)

unique_tickers_with_data = prices.index.get_level_values("ticker").unique()
print(f"\nTickers with data: {len(unique_tickers_with_data)} / {len(all_tickers)}")
print(f"Date range: {prices.index.get_level_values('date').min()} → {prices.index.get_level_values('date').max()}")
print(f"Total bars: {len(prices):,}")

# Sanity
assert len(unique_tickers_with_data) > 3000, "Too few tickers got price data"


Total tickers to download: 5699



Tickers with data: 5553 / 5699
Date range: 2010-01-04 00:00:00 → 2026-05-07 00:00:00
Total bars: 14,706,606


In [5]:
# === Cell 4: 市场上下文（^GSPC / ^VIX / 11 SPDR ETF）===
MARKET_CACHE = CACHE_DIR / "market_data.parquet"

if REFRESH_DATA or not MARKET_CACHE.exists():
    market = download_prices(
        MARKET_TICKERS,
        start=HISTORY_START.strftime("%Y-%m-%d"),
        end=(ANCHOR_DATE + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    )
    market = market.set_index(["ticker", "date"]).sort_index()
    market.to_parquet(MARKET_CACHE)
else:
    market = pd.read_parquet(MARKET_CACHE)

market_tickers = market.index.get_level_values("ticker").unique()
print(f"Market tickers loaded: {len(market_tickers)} / {len(MARKET_TICKERS)}")
assert "^GSPC" in market_tickers and "^VIX" in market_tickers, "SPX/VIX missing"
assert len(market_tickers) >= 12, "Some sector ETFs missing"


Market tickers loaded: 13 / 13
